# Study 1 — 01d: Dependency Analysis

Analyses the `depends_on` graph structure: coverage, distance statistics, and hypothesis cycle identification.

**Outputs:**
- `outputs/study1_analysis/tables/dependency_statistics.csv`
- `outputs/study1_analysis/tables/hypothesis_cycles.csv`
- `outputs/study1_analysis/figures/dependency_distance_histogram.png`
- `outputs/study1_analysis/figures/dependency_distance_by_label.png`
- `outputs/study1_analysis/figures/cycle_length_distribution.png`
- `outputs/study1_analysis/figures/cycles_per_task.png`

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
from study1_helpers import *

traces = load_traces()
df = build_sentence_df(traces)
print(f'Loaded: {df.trace_key.nunique()} traces, {len(df):,} sentences')

## 4a. Dependency Coverage

In [ ]:
section_header('4a. Dependency Coverage')

has_dep = df['n_dependencies'] > 0
pct_overall = has_dep.mean() * 100
print(f'Overall dependency coverage: {has_dep.sum()}/{len(df)} ({pct_overall:.1f}%)')

# Per micro-label
dep_by_label = df.groupby('micro_label').apply(
    lambda g: pd.Series({
        'total': len(g),
        'with_dep': (g['n_dependencies'] > 0).sum(),
        'empty': (g['n_dependencies'] == 0).sum(),
        'pct_with_dep': (g['n_dependencies'] > 0).mean() * 100
    })
).reindex(MICRO_LABELS)
print('\nDependency coverage by micro-label:')
display(dep_by_label.round(1))

# Flag traces below 90% (excluding ORIENT)
non_orient = df[df['micro_label'] != 'ORIENT']
trace_coverage = non_orient.groupby('trace_key').apply(
    lambda g: (g['n_dependencies'] > 0).mean() * 100
)
low_cov = trace_coverage[trace_coverage < 90]
if len(low_cov) > 0:
    print(f'\nTraces below 90% coverage (excluding ORIENT): {len(low_cov)}')
    display(low_cov.sort_values())
else:
    print(f'\nAll traces have ≥90% coverage (excluding ORIENT).')

dep_stats_rows = []
for m in MICRO_LABELS:
    row = dep_by_label.loc[m]
    dep_stats_rows.append({
        'label': m, 'total': int(row['total']), 'with_dep': int(row['with_dep']),
        'empty': int(row['empty']), 'pct_with_dep': round(row['pct_with_dep'], 1)
    })

## 4b. Dependency Graph Statistics

In [ ]:
section_header('4b. Dependency Graph Statistics')

# Mean dependency count per sentence
mean_deps_overall = df['n_dependencies'].mean()
print(f'Mean dependencies per sentence (overall): {mean_deps_overall:.2f}')

mean_deps_by_label = df.groupby('micro_label')['n_dependencies'].mean().reindex(MICRO_LABELS)
print('\nMean dependencies per sentence by label:')
display(mean_deps_by_label.round(2))

# Dependency distance
distances = []
for _, row in df.iterrows():
    sid = row['sentence_id']
    for dep in row['depends_on']:
        dist = abs(sid - dep)
        distances.append({
            'trace_key': row['trace_key'],
            'micro_label': row['micro_label'],
            'sentence_id': sid,
            'dep_id': dep,
            'distance': dist,
        })
dist_df = pd.DataFrame(distances)
print(f'\nTotal dependency edges: {len(dist_df):,}')
print(f'Distance: mean={dist_df["distance"].mean():.1f}, '
      f'median={dist_df["distance"].median():.0f}, '
      f'95th pctile={dist_df["distance"].quantile(0.95):.0f}')

# Distance by label
dist_by_label = dist_df.groupby('micro_label')['distance'].agg(['mean', 'median']).reindex(MICRO_LABELS)
print('\nMean dependency distance by label:')
display(dist_by_label.round(1))

In [ ]:
# Visualise: dependency distance histogram
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(dist_df['distance'].clip(upper=100), bins=50, color='#4C72B0', edgecolor='white', alpha=0.8)
ax.set_xlabel('Dependency distance (sentences)')
ax.set_ylabel('Count')
ax.set_title('Dependency Distance Distribution (clipped at 100)')
ax.axvline(x=dist_df['distance'].median(), color='red', linestyle='--', label=f'Median={dist_df["distance"].median():.0f}')
ax.legend()
plt.tight_layout()
save_fig(fig, 'dependency_distance_histogram.png')
plt.show()

# Visualise: mean distance by label
fig, ax = plt.subplots(figsize=(10, 5))
colours = [MICRO_COLOURS[m] for m in MICRO_LABELS]
ax.bar(MICRO_LABELS, dist_by_label['mean'].values, color=colours, edgecolor='white')
ax.set_ylabel('Mean dependency distance')
ax.set_xlabel('Micro-label')
ax.set_title('Mean Dependency Distance by Micro-label')
ax.set_xticklabels(MICRO_LABELS, rotation=45, ha='right')
plt.tight_layout()
save_fig(fig, 'dependency_distance_by_label.png')
plt.show()

## 4c. Hypothesis Cycle Structure

In [ ]:
section_header('4c. Hypothesis Cycle Structure')

def find_cycles(trace_df):
    """
    Identify hypothesis cycles: HYPO → (TEST)* → JUDGE sequences using
    the dependency graph. A cycle starts at a HYPO and ends at the first
    JUDGE whose depends_on includes that HYPO.
    """
    cycles = []
    trace_df = trace_df.sort_values('sentence_id').reset_index(drop=True)

    hypo_sids = trace_df.loc[trace_df['micro_label'] == 'HYPO', 'sentence_id'].tolist()

    for hypo_sid in hypo_sids:
        # Find JUDGEs that depend on this HYPO
        judges = trace_df[
            (trace_df['micro_label'] == 'JUDGE') &
            (trace_df['sentence_id'] > hypo_sid) &
            (trace_df['depends_on'].apply(lambda deps: hypo_sid in deps))
        ]
        if len(judges) > 0:
            judge_sid = judges.iloc[0]['sentence_id']
            cycle_sents = trace_df[
                (trace_df['sentence_id'] >= hypo_sid) &
                (trace_df['sentence_id'] <= judge_sid)
            ]
            judge_row = judges.iloc[0]
            cycles.append({
                'hypo_sid': hypo_sid,
                'judge_sid': judge_sid,
                'length': len(cycle_sents),
                'judgement': judge_row.get('judgement', None),
            })
    return cycles

# Find all cycles across all traces
all_cycles = []
for key, grp in df.groupby('trace_key'):
    task_id = grp['task_id'].iloc[0]
    trace_cycles = find_cycles(grp)
    for c in trace_cycles:
        c['trace_key'] = key
        c['task_id'] = task_id
    all_cycles.extend(trace_cycles)

cycles_df = pd.DataFrame(all_cycles)
print(f'Total hypothesis cycles found: {len(cycles_df)}')

# Cycles per trace
cycles_per_trace = df.groupby('trace_key').apply(
    lambda g: len(find_cycles(g))
)
print(f'Cycles per trace: mean={cycles_per_trace.mean():.1f}, median={cycles_per_trace.median():.0f}')

# Cycle length
if len(cycles_df) > 0:
    print(f'Cycle length: mean={cycles_df["length"].mean():.1f}, median={cycles_df["length"].median():.0f}')
    print(f'  Short cycles (≤3): {(cycles_df["length"] <= 3).sum()} ({(cycles_df["length"] <= 3).mean()*100:.1f}%)')
    print(f'  Long cycles (≥10): {(cycles_df["length"] >= 10).sum()} ({(cycles_df["length"] >= 10).mean()*100:.1f}%)')

In [ ]:
# Cycle completion rate: % of HYPO sentences that reach a JUDGE
n_hypo = (df['micro_label'] == 'HYPO').sum()
n_hypo_with_judge = cycles_df['hypo_sid'].nunique() if len(cycles_df) > 0 else 0
completion_rate = n_hypo_with_judge / n_hypo * 100 if n_hypo > 0 else 0
print(f'Cycle completion rate: {n_hypo_with_judge}/{n_hypo} HYPOs reach a JUDGE ({completion_rate:.1f}%)')

# Cycle outcome for completed cycles
if len(cycles_df) > 0:
    outcome = cycles_df['judgement'].value_counts()
    outcome_pct = (outcome / len(cycles_df) * 100).round(1)
    print(f'\nCycle outcomes:')
    for k, v in outcome.items():
        print(f'  {k}: {v} ({outcome_pct[k]}%)')

# Stratify by task
if len(cycles_df) > 0:
    print('\nCycles per task:')
    task_cycles = cycles_df.groupby('task_id').agg(
        n_cycles=('length', 'size'),
        mean_length=('length', 'mean'),
        median_length=('length', 'median'),
    )
    display(task_cycles.round(1))

In [ ]:
# Visualise: cycle length distribution
if len(cycles_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Cycle length histogram
    axes[0].hist(cycles_df['length'].clip(upper=30), bins=30, color='#C44E52', edgecolor='white', alpha=0.8)
    axes[0].set_xlabel('Cycle length (sentences)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Hypothesis Cycle Length Distribution')
    axes[0].axvline(x=cycles_df['length'].median(), color='black', linestyle='--',
                    label=f'Median={cycles_df["length"].median():.0f}')
    axes[0].legend()

    # Mean cycles per task
    task_means = cycles_per_trace.reset_index()
    task_means.columns = ['trace_key', 'n_cycles']
    task_means['task_id'] = task_means['trace_key'].str.extract(r'task(\d)').astype(int)
    task_summary = task_means.groupby('task_id')['n_cycles'].mean()
    axes[1].bar(task_summary.index, task_summary.values, color='#8172B2', edgecolor='white')
    axes[1].set_xlabel('Task')
    axes[1].set_ylabel('Mean cycles per trace')
    axes[1].set_title('Mean Hypothesis Cycles per Task')

    plt.tight_layout()
    save_fig(fig, 'cycle_length_distribution.png', close=False)
    # Also save the second panel separately
    plt.show()

# Save tables
dep_stats_df = pd.DataFrame(dep_stats_rows)
dep_stats_df['mean_deps'] = mean_deps_by_label.values
dep_stats_df['mean_distance'] = dist_by_label['mean'].values
save_table(dep_stats_df, 'dependency_statistics.csv')

if len(cycles_df) > 0:
    save_table(cycles_df, 'hypothesis_cycles.csv')
    print(f'\nSaved {len(cycles_df)} cycles.')